# Maternal Health Risk Prediction

This notebook is part of a GitHub portfolio project that predicts maternal health risk level using machine learning.

The project is for learning and portfolio demonstration only. It is not a medical diagnostic tool.


Prepare & Clean Data
🔧 What you MUST do in code **bold text**

**Load data**

In [ ]:
import pandas as pd

df = pd.read_csv("data/Maternal_Health_Risk_Data_Set_Modified.csv")
df.head()

**Remove duplicates**

In [ ]:
df = df.drop_duplicates()

**Handle missing values (median)**

In [ ]:
df = df.fillna(df.median(numeric_only=True))

Normalize data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = df.drop(["RiskLevel", "CitizenID"], axis=1)

X_scaled = scaler.fit_transform(X)

import pandas as pd
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
X_scaled_df.describe().round(2)

In [ ]:
print(df["RiskLevel"].unique())

In [ ]:
df["RiskLevel"] = df["RiskLevel"].astype(str).str.strip().str.lower()

In [ ]:
df["RiskLevel"] = df["RiskLevel"].map({
    "low risk": 0,
    "high risk": 1
})

In [ ]:
print(df["RiskLevel"].isna().sum())
print(df["RiskLevel"].unique())

In [ ]:
from sklearn.model_selection import train_test_split
y = df["RiskLevel"]

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
print("Train distribution:\n", y_train.value_counts())
print("Test distribution:\n", y_test.value_counts())

**Random Forest Model**

Train-test split (IMPORTANT SETTINGS)

Train Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

Evaluate model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

**ROC Curve**

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_probs = rf_model.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

**Feature importance**

In [ ]:
import pandas as pd

importance = pd.Series(rf_model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance.head(3))

**Neural Network**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import pandas as pd

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)

In [ ]:
X_train_scaled_df.describe().round(2)

**Build model**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

dnn = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

dnn.fit(X_train_scaled, y_train, epochs=50, verbose=0)

**Evaluate**

In [ ]:
y_pred_dnn = (dnn.predict(X_test_scaled) > 0.5).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred_dnn))
print("Precision:", precision_score(y_test, y_pred_dnn))
print("Recall:", recall_score(y_test, y_pred_dnn))
print("F1:", f1_score(y_test, y_pred_dnn))

Save model

In [ ]:
import joblib

joblib.dump(rf_model, "models/risk_model.joblib")

Gradio APP

In [ ]:
import gradio as gr
import joblib
import numpy as np

model = joblib.load("models/risk_model.joblib")

def predict(age, sysbp, diabp, bs, temp, hr):
    data = np.array([[age, sysbp, diabp, bs, temp, hr]])
    pred = model.predict(data)
    return "High Risk" if pred[0] == 1 else "Low Risk"

app = gr.Interface(
    fn=predict,
    inputs=["number","number","number","number","number","number"],
    outputs="text"
)

app.launch()